In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

# 1. Load dataset
df = pd.read_csv("Lipstick.csv")
print("Original data:\n", df, "\n")

# 2. Encode categorical columns (one encoder per column)
label_encoders = {}
for col in df.columns:
    if df[col].dtype == 'object' or df[col].dtype.name == 'category':
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

# Optional: print mappings for viva
print("Mappings (class -> numeric):")
for col, le in label_encoders.items():
    print(f" {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print()

# 3. Split into features and target
X = df[['Age', 'Income', 'Gender', 'Ms']]
y = df['Buys']

# 4. Train Decision Tree
model = DecisionTreeClassifier(criterion="entropy", random_state=42)
model.fit(X, y)
print("Model trained.\n")

# 5. Test sample for Practical 7 (raw categorical values)
test_raw = {'Age': '>35', 'Income': 'Medium', 'Gender': 'Female', 'Ms': 'Married'}
print("Raw test sample:", test_raw)

# 6. Encode the test sample using saved encoders (transform, not fit)
encoded_test = []
for col in X.columns:
    le = label_encoders.get(col)
    if le is None:
        raise ValueError(f"No encoder for column '{col}'")
    val = str(test_raw[col])
    if val not in le.classes_:
        raise ValueError(f"Unseen category '{val}' for column '{col}'. Allowed: {list(le.classes_)}")
    encoded_test.append(int(le.transform([val])[0]))

print("Encoded test vector (in X.columns order):", encoded_test)

# 7. Predict — pass DataFrame with same column names to avoid warning
test_df = pd.DataFrame([encoded_test], columns=X.columns)
pred_num = model.predict(test_df)[0]
pred_label = label_encoders['Buys'].inverse_transform([pred_num])[0]

print("\nPrediction (numeric):", pred_num)
print("Final Decision (decoded):", pred_label)



Original data:
     Id    Age  Income  Gender       Ms Buys
0    1    <21    High    Male   Single   No
1    2    <21    High    Male  Married   No
2    3  21-35    High    Male   Single  Yes
3    4    >35  Medium    Male   Single  Yes
4    5    >35     Low  Female   Single  Yes
5    6    >35     Low  Female  Married   No
6    7  21-35     Low  Female  Married  Yes
7    8    <21  Medium    Male   Single   No
8    9    <21     Low  Female  Married  Yes
9   10    >35  Medium  Female   Single  Yes
10  11    <21  Medium  Female  Married  Yes
11  12  21-35  Medium    Male  Married  Yes
12  13  21-35    High  Female   Single  Yes
13  14    >35  Medium    Male  Married   No 

Mappings (class -> numeric):
 Age: {'21-35': np.int64(0), '<21': np.int64(1), '>35': np.int64(2)}
 Income: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}
 Gender: {'Female': np.int64(0), 'Male': np.int64(1)}
 Ms: {'Married': np.int64(0), 'Single': np.int64(1)}
 Buys: {'No': np.int64(0), 'Yes': np.int64(